<div align="center">
<img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" width="320" alt="Wayne State University">
<h1>Practice 5 — Optional Lab Follow-up</h1></div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week05/practice05_lab_followup.ipynb)

ME 5995 — AI in Mechanical Engineering I · Optional, ungraded · No submission

[Course repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) ·
[Practice index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb)

Run this notebook in Colab independently of Lab 5. Setup repeats the same CSV,
11 sensor features, train-only scaling and c1/c4 split. c6 is not fitted or evaluated.
Complete the activities you want; compare with the companion self-check solution.

## 1. Lower the decision threshold

The supplied example uses 0.50. Predict what lowering it to 0.25 does to false
alarms, then change one value and check the output. No new model is trained.

**Version:** Student starter

[Starter](practice05_lab_followup.ipynb) · [Self-check solution](practice05_lab_followup_solution.ipynb) — try before checking.

In [ ]:
# PROVIDED — RUN UNCHANGED. Imports do not fit a model.
import numpy as np  # Compare arrays of labels during the data check.
import pandas as pd  # Read CSV files and work with tables.
import matplotlib.pyplot as plt  # Draw figures.
from IPython.display import display  # Show a table in the notebook.
# Import each tool needed to split data, scale inputs and train classifiers.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

plt.rcParams['font.size'] = 11  # Set the default text size in plots.
plt.rcParams['figure.dpi'] = 110  # Set the plot display resolution.

In [ ]:
# PROVIDED — RUN UNCHANGED. Load the course CSV directly in Colab.
data_url = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv'
data = pd.read_csv(data_url)  # Internet access is required; no file selection.

# Detect an older CSV rather than silently using the earlier 50 µm boundary.
# Start a separate check column; do not overwrite the supplied labels.
expected_level = pd.Series(0, index=data.index)
# Values above 75 micrometers are at least Intermediate-wear.
expected_level.loc[data['wear_mean_um'] > 75] = 1
# Values at or above 150 micrometers belong to Severe-wear instead.
expected_level.loc[data['wear_mean_um'] >= 150] = 2
if not np.array_equal(data['wear_level'], expected_level):
    raise ValueError('This CSV has outdated/inconsistent wear labels. Download the current course CSV (75/150 µm).')
print('Rows and columns:', data.shape)
display(data[['cutter_id', 'cut_number', 'wear_mean_um', 'wear_level']].head(3))

In [ ]:
# PROVIDED — RUN UNCHANGED. One split for all models and both tasks.
# True identifies a row from either development cutter.
development_rows = data['cutter_id'].isin(['c1', 'c4'])
# .loc selects those rows; .copy creates a separate working table.
development = data.loc[development_rows].copy()
# Keep all c6 rows apart until final evaluation.
test_rows = data['cutter_id'] == 'c6'
test = data.loc[test_rows].copy()
# Convert the class number to text, then form groups such as 'c1_0'.
class_text = development['wear_level'].astype(str)
strata = development['cutter_id'] + '_' + class_text
# Reserve 20% for validation. Seed 42 reproduces the same split.
# stratify keeps each cutter/class group represented in both partitions.
train, valid = train_test_split(development, test_size=0.20,
                               random_state=42, stratify=strata)
# Sorting only makes output readable; it does not change partition membership.
train = train.sort_index()
valid = valid.sort_index()
print('Train / validation / test rows:', len(train), len(valid), len(test))
print('Three-class counts: columns 0=Slight, 1=Intermediate, 2=Severe')
print('Training counts by cutter:')
display(pd.crosstab(train['cutter_id'], train['wear_level']))
print('Validation counts by cutter:')
display(pd.crosstab(valid['cutter_id'], valid['wear_level']))

In [ ]:
# PROVIDED — RUN UNCHANGED. Explicit sensor inputs, identical to Lab 4.
sensor_features = [
    'force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd',
    'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd',
    'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd',
]
X_train = train[sensor_features]  # Training rows, sensor columns only.
X_valid = valid[sensor_features]  # Validation rows, the same columns/order.
y_train_multi = train['wear_level']  # Original training labels: 0, 1 or 2.
y_valid_multi = valid['wear_level']  # Actual validation labels: 0, 1 or 2.
# Ask whether each cut has Severe wear; the result is True or False.
train_is_severe = y_train_multi == 2
valid_is_severe = y_valid_multi == 2
# Convert True to 1 and False to 0 for the binary task.
y_train_binary = train_is_severe.astype(int)
# Convert each Boolean decision into a class label: True becomes 1, False becomes 0.
y_valid_binary = valid_is_severe.astype(int)

scaler = StandardScaler()  # Create an unfitted scaling tool.
scaler.fit(X_train)  # Learn one mean and standard deviation per training column.
X_train_scaled = scaler.transform(X_train)  # Apply (value - mean) / SD.
X_valid_scaled = scaler.transform(X_valid)  # Reuse TRAINING means/SDs; do not fit again.
print('Training input shape:', X_train_scaled.shape)

In [ ]:
# GUIDED — RUN UNCHANGED. Completed fit/predict examples.
binary_baseline = DummyClassifier(strategy='most_frequent')  # Always use the common class.
binary_baseline.fit(X_train_scaled, y_train_binary)  # Find the common TRAINING label.
baseline_binary_pred = binary_baseline.predict(X_valid_scaled)  # One label per validation cut.

# Create a model. C controls regularization; keep the supplied value.
# lbfgs is the fitting algorithm; max_iter limits its optimization iterations.
logistic_binary = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
logistic_binary.fit(X_train_scaled, y_train_binary)  # Learn from training inputs and labels.
logistic_binary_pred = logistic_binary.predict(X_valid_scaled)  # Predict unseen validation rows.
print('Baseline accuracy:', round(accuracy_score(y_valid_binary, baseline_binary_pred), 3))
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
print('Baseline Severe F1:', round(f1_score(y_valid_binary, baseline_binary_pred, zero_division=0), 3))

In [ ]:
# PROVIDED EXAMPLE — predict Severe using a threshold of 0.50.
class_probabilities = logistic_binary.predict_proba(X_valid_scaled)
severe_probability = class_probabilities[:, 1]  # Column 1 is P(Severe).
example_threshold = 0.50
# The comparison is True where the estimated positive-class probability reaches the cutoff.
pred_middle = (severe_probability >= example_threshold).astype(int)
# Count actual labels by row and predicted labels by column, in the specified label order.
matrix_middle = confusion_matrix(y_valid_binary, pred_middle, labels=[0, 1])
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
example_f1 = f1_score(y_valid_binary, pred_middle, zero_division=0)

In [ ]:
# REQUIRED — change only the threshold; keep the same model and probabilities.
student_threshold = 0.50  # TODO: Change 0.50 to 0.25 after recording your expectation.
# PROVIDED — True becomes 1 (Severe); False becomes 0 (Normal).
pred_high = (severe_probability >= student_threshold).astype(int)
# Count actual labels by row and predicted labels by column, in the specified label order.
matrix_high = confusion_matrix(y_valid_binary, pred_high, labels=[0, 1])
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
student_f1 = f1_score(y_valid_binary, pred_high, zero_division=0)

# Compare the two rules. Matrix [0, 1] counts FP; [1, 0] counts FN.
threshold_results = pd.DataFrame()
threshold_results['Threshold'] = [example_threshold, student_threshold]
threshold_results['False alarms (FP)'] = [matrix_middle[0, 1], matrix_high[0, 1]]
threshold_results['Missed Severe (FN)'] = [matrix_middle[1, 0], matrix_high[1, 0]]
threshold_results['Severe F1'] = [example_f1, student_f1]
display(threshold_results)

# A successful run does not mean the requested change has been completed.
if student_threshold != 0.25:
    print('Practice edit is unfinished: change student_threshold to 0.25 and rerun this cell.')

**Your observation:** FP changed from ___ to ___. Explain why a lower threshold need not improve F1.

## 2. Compare SVM boundaries

These supplied results use force x SD and vibration x SD, scaled using training
rows only. They are separate two-feature models, not a projection of the Lab's
11-feature models. Both use the same c1/c4 train/validation rows; no c6 is evaluated.
Solid lines are boundaries; dashed lines are linear margins; rings are support vectors.

![Two-feature PHM SVM comparison](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/practice05/phm_two_feature_svm.png)

Does the curved boundary give a better validation F1 here? Cite the scores.
Curves in empty regions are not verified physical relationships.

## 3. Interpret labels after regrouping

Suppose a three-class model predicts Slight for an actual Intermediate cut.
If both labels are mapped to binary Normal, is that particular prediction still
an error? This is label regrouping, not retraining a binary model.

**Your answer:** ___

## References

- [PHM Society 2010 Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/)
- [Course feature provenance and instructional labels](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md)

Wear categories are course-derived, not industrial replacement limits.